In [0]:
# === CONFIGURATION ===
import os

# Same workspace - derive HOST automatically from notebook context
HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

# Replace with your SP credentials
CLIENT_ID = "6d414e48-bc10-44fd-901e-3cff5a51dfb7"              # SP application (client) ID
CLIENT_SECRET = "redacted-after-execution"      # SP OAuth secret

# Existing endpoint to host the test index
ENDPOINT_NAME = "kishor-test"

# Test index (Direct Access - simplest for repro, no source table needed)
TEST_INDEX_NAME = "kishor_test.models.sp_oauth_repro_test_index"

print(f"Workspace: {HOST}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Test index: {TEST_INDEX_NAME}")

In [0]:
"""Create a small Direct Access vector index for the repro.
This cell uses the notebook's attached credentials (PAT or cluster identity).
Direct Access = no source table needed; we upsert vectors manually.
"""
import json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import DirectAccessVectorIndexSpec, EmbeddingVectorColumn, VectorIndexType

w = WorkspaceClient()  # uses notebook-attached creds

# Create a tiny Direct Access index (3 dimensions for minimal test)
try:
    index = w.vector_search_indexes.create_index(
        name=TEST_INDEX_NAME,
        endpoint_name=ENDPOINT_NAME,
        primary_key="id",
        index_type=VectorIndexType.DIRECT_ACCESS,
        direct_access_index_spec=DirectAccessVectorIndexSpec(
            embedding_vector_columns=[
                EmbeddingVectorColumn(name="embedding", embedding_dimension=3)
            ],
            schema_json=json.dumps({
                "id": "string",
                "text": "string",
                "embedding": "array<float>"
            })
        ),
    )
    print(f"Index created: {TEST_INDEX_NAME}")
    print(f"Status: {index.status}")
except Exception as e:
    if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
        print(f"Index already exists: {TEST_INDEX_NAME} - skipping creation")
    else:
        raise

In [0]:
"""Upsert a few test vectors so we have data to query."""
import json, time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Wait briefly for index to be ready if just created
time.sleep(5)

test_data = [
    {"id": "doc1", "text": "Hello world",      "embedding": [0.1, 0.2, 0.3]},
    {"id": "doc2", "text": "Machine learning", "embedding": [0.4, 0.5, 0.6]},
    {"id": "doc3", "text": "Vector search",    "embedding": [0.7, 0.8, 0.9]},
]

result = w.vector_search_indexes.upsert_data_vector_index(
    index_name=TEST_INDEX_NAME,
    inputs_json=json.dumps(test_data)
)
print(f"Upsert result: {result}")

In [0]:
%pip install databricks-vectorsearch -q

In [0]:
"""This is the actual repro cell.
Uses databricks.vector_search.client.VectorSearchClient (aka AISearchClient)
with Service Principal OAuth credentials.
"""
from databricks.vector_search.client import VectorSearchClient

# Authenticate with Service Principal credentials (OAuth M2M)
vsc = VectorSearchClient(
    workspace_url=HOST,
    service_principal_client_id=CLIENT_ID,
    service_principal_client_secret=CLIENT_SECRET,
)

# Get the index handle
index = vsc.get_index(
    endpoint_name=ENDPOINT_NAME,
    index_name=TEST_INDEX_NAME,
)

print(f"Index description: {index.describe()}")
print("---")

# Similarity search using a query vector
results = index.similarity_search(
    query_vector=[0.1, 0.2, 0.3],
    columns=["id", "text"],
    num_results=3,
)

print("Similarity search results:")
for row in results.get("result", {}).get("data_array", []):
    print(f"  {row}")

In [0]:
"""Test VectorSearchClient using notebook PAT/token directly."""
from databricks.vector_search.client import VectorSearchClient

TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

vsc = VectorSearchClient(
    workspace_url=HOST,
    personal_access_token=TOKEN,
)

index = vsc.get_index(
    endpoint_name=ENDPOINT_NAME,
    index_name=TEST_INDEX_NAME,
)

print(f"Index description: {index.describe()}")
print("---")

results = index.similarity_search(
    query_vector=[0.1, 0.2, 0.3],
    columns=["id", "text"],
    num_results=3,
)

print("Similarity search results:")
for row in results.get("result", {}).get("data_array", []):
    print(f"  {row}")